# 43008: Reinforcement Learning - WOW Naval MAPPO with a Fleet Commander

## Imitation: copy the Elite rule AI before reinforcement learning

**Imitation** · [Stage 0](Stage0-Gunnery.ipynb) · [Stage 1](Stage1-Capture.ipynb) · [Stage 2](Stage2-Defend.ipynb) · [Stage 3](Stage3-Duel.ipynb) · [Stage 4](Stage4-King-of-the-Hill.ipynb) · [Stage 5](Stage5-Archipelago.ipynb) · [Stage 6](Stage6-Domination.ipynb) · [Stage 7](Stage7-Open.ipynb) · [the full walkthrough](WOW-MAPPO-Simple.ipynb)

A random policy almost never wins a naval battle, so reinforcement learning from scratch spends most of its time discovering what "sail, aim, shoot" means. AlphaStar (StarCraft II) solved the same problem by **first copying human players with supervised learning, then improving with RL**. We have no human replays, but the game has an **Elite rule-based AI** - so we copy that.

1. **Record** - the Elite AI flies both fleets; for every decision of every one of our ships the game reports the option it chose on each of the six action heads. That is a labelled dataset: *observation → action*.
2. **Clone** - train the captain network to predict those labels. It is **Week 9 Part A** again: a classifier, a `DataLoader`, `CrossEntropyLoss` (one per action head), `Adam`, a loss curve and an accuracy curve.
3. **Hand over** - the cloned captain is saved as `models/imitation.pt`. The Stage 0 notebook starts the RL model from it, and a KL penalty that fades out keeps the captain close to the clone at first (AlphaStar's KL to the supervised policy).

In this project's first pipeline, cloning alone won **58%** of King of the Hill battles where RL from scratch won 20%.

Run this notebook **once per run, before Stage 0**.

### The method in one picture

| Piece | What it does | Where it comes from | Course link |
|---|---|---|---|
| **Captain** (every ship, one shared network) | its own view → 6 action heads; a transformer over ships/circles/islands, a GRU memory, pointer heads for *which target* and *which circle* | AlphaStar, OpenAI Five | W10-C custom feature extractor; frame stacking → memory |
| **Fleet commander** | every 10 s: an order per ship - free / engage / hold circle k | Honor of Kings' hierarchical macro strategy | the same PPO at a slower clock |
| **Centralised critic** | sees the true battle (training only); one value per reward group + win probability | MAPPO, Honor of Kings multi-head value | W10-A/B actor-critic |
| **Update** | PPO with dual clip, value normalisation and clipping | MAPPO, Honor of Kings | W10-B PPO |
| **Warm start** | the captain first copies the Elite rule AI (StageP), then a fading KL penalty keeps it close | AlphaStar | W9-A supervised training loop |
| **League** | the enemy is the rule AI, our latest copy, or a past copy (PFSP) - with Elo ratings | OpenAI Five, AlphaStar | W9-C/D frozen target network |
| **Curriculum** | 8 stages, each passed at ≥70% of its own goal; earlier stages are replayed 10-15% of the time | TiZero, OpenAI Five | - |

## Setup

In [ ]:
import os, time, zipfile
# SageMaker Studio (/home/sagemaker-user) or a SageMaker notebook instance (/home/ec2-user/SageMaker)
ON_SAGEMAKER = any(os.path.isdir(p) for p in ("/home/sagemaker-user", "/home/ec2-user/SageMaker"))
home = os.path.expanduser("~/SageMaker") if os.path.isdir(os.path.expanduser("~/SageMaker")) else os.path.expanduser("~")
bundle = os.path.join(home, "wow_sagemaker.zip")
if ON_SAGEMAKER and os.path.basename(os.getcwd()) != "Training":
    if not os.path.isdir(os.path.join(home, "wow")):
        assert os.path.exists(bundle), f"upload wow_sagemaker.zip to {home} first"
        zipfile.ZipFile(bundle).extractall(os.path.join(home, "wow"))
    %cd {home}/wow/Training

# Uploading a new wow_sagemaker.zip does not replace the unzipped folder: check which version runs here
running = open("BUNDLE_VERSION").read().strip() if os.path.exists("BUNDLE_VERSION") else "an older one"
if ON_SAGEMAKER and os.path.exists(bundle):
    with zipfile.ZipFile(bundle) as z:
        uploaded = z.read("Training/BUNDLE_VERSION").decode().strip() if "Training/BUNDLE_VERSION" in z.namelist() else "unknown"
    if uploaded != running:
        print(f"*** {bundle} is version {uploaded}, but this notebook runs version {running}. Unzip it into a new folder "
              f"and open this notebook from there. ***")
print(os.getcwd(), "| bundle", running, "| glibc", os.confstr("CS_GNU_LIBC_VERSION"))

In [ ]:
# install only what is missing (PyTorch as the CPU build)
import importlib.util
missing = [pkg for mod, pkg in (("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("PIL", "pillow"))
           if importlib.util.find_spec(mod) is None]
if missing:
    !pip install -q {" ".join(missing)}
if importlib.util.find_spec("torch") is None:
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
import os
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from simple_mappo import (NavalEnv, MAPPO, TeamStorage, DEFAULT_CURRICULUM, collect_expert, save_dataset, load_dataset,
                          behaviour_cloning, save_imitation, plot_imitation, evaluate)

## Team checkpoints

The same `RUN_NAME` as the stage notebooks: the imitation checkpoint goes into that run's folder, where Stage 0 picks it up.

In [ ]:
YOUR_NAME = "your-name"              # shown to your teammates
RUN_NAME = "fleet"                   # the same in every stage notebook (and StageP)
if ON_SAGEMAKER:
    from simple_mappo.aws import team_storage_uri
    TEAM_STORAGE = team_storage_uri()     # s3://sagemaker-<region>-<account>/wow-mappo, or "s3://<team-bucket>/wow-mappo"
else:
    TEAM_STORAGE = None              # e.g. "s3://<team-bucket>/wow-mappo"; None trains on this machine only

storage = TeamStorage(TEAM_STORAGE, RUN_NAME, user=YOUR_NAME) if TEAM_STORAGE else None
run_dir = f"runs_simple/{RUN_NAME}"
log_dir, models_dir, video_dir = (os.path.join(run_dir, d) for d in ("logs", "models", "videos"))
os.makedirs(models_dir, exist_ok=True)
if storage:
    storage.show_status()

if storage:
    storage.pull(run_dir)            # the run's files so far (and its imitation checkpoint, if a teammate made one)
if os.path.exists(os.path.join(models_dir, "imitation.pt")):
    print("this run already has an imitation checkpoint (models/imitation.pt) - running on replaces it")

## Configuration

The network settings (`d_model`, `attn_layers`, `hidden_size`, `recurrent`) must be **the same as in the stage notebooks** - the clone becomes the RL model's captain.

In [ ]:
config = OrderedDict([
    ('n_envs', max(1, min(16, os.cpu_count() // 2))),     # battles recorded in parallel
    ('n_battles', 60),                 # battles of the Elite AI to record
    ('record_stages', [2, 3, 4, 5, 6]),  # curriculum stages to record (both fleets: the Elite AI)
    ('num_epochs', 8),                 # passes over the recorded data
    ('batch_size', 64),                # windows per minibatch
    ('window', 32),                    # decisions per window (the GRU learns from sequences)
    ('learning_rate', 3e-4),
])
mappo_settings = dict(learning_rate=3e-4, critic_learning_rate=5e-4, n_steps=256, n_minibatches=4, n_epochs=5,
                      gamma=0.995, gae_lambda=0.95, clip_range=0.2, dual_clip=3.0, ent_coef=0.01,
                      d_model=64, attn_layers=2, hidden_size=128, recurrent=True,      # the networks
                      commander=True, commander_period=10,                             # the fleet commander
                      kl_bc_coef=0.1, opponent_refresh=10)                             # imitation anchor, league
pd.Series(config).to_frame("value")

## Step 1: record the Elite AI

Each battle slot draws one of `record_stages`; both fleets are the Elite AI, and our fleet's decisions are labelled. About a minute per 10 battles with 8 battle slots.

In [ ]:
recorder = NavalEnv(stages="default", n_envs=config['n_envs'], record_expert=True, base_port=5300,
                    unity_log_dir=os.path.join(run_dir, "unity_logs"))
data = collect_expert(recorder, config['n_battles'], stages=config['record_stages'])
dims, heads = recorder.dims, recorder.heads
recorder.close()
save_dataset(data, os.path.join(run_dir, "imitation", "demos.npz"))

### Look at the data

What the Elite AI chooses on each head - like looking at the Fashion-MNIST samples before training. A head where one option dominates is easy to copy; the interesting ones are *move* and *target*.

In [ ]:
names = [h["name"] for h in heads]
fig, axes = plt.subplots(1, len(names), figsize=(18, 3))
for h, (ax, name) in enumerate(zip(axes, names)):
    labels = data["labels"][data["valid"], h]
    ax.bar(*np.unique(labels, return_counts=True))
    ax.set_title(name)
    ax.set_xlabel("option")
plt.tight_layout()
plt.show()
print(f"{len(data['lengths'])} ship sequences, {int(data['valid'].sum()):,} labelled decisions")

## Step 2: clone it

The Week 9 Part A loop, with one cross-entropy per action head. Held-out accuracy is measured on ships the training never saw.

In [ ]:
captain, history = behaviour_cloning(data, dims, heads, d_model=mappo_settings['d_model'],
                                     attn_layers=mappo_settings['attn_layers'], hidden_size=mappo_settings['hidden_size'],
                                     recurrent=mappo_settings['recurrent'], num_epochs=config['num_epochs'],
                                     batch_size=config['batch_size'], window=config['window'],
                                     learning_rate=config['learning_rate'])
plot_imitation(history, names)

## Step 3: save and hand over

In [ ]:
path = save_imitation(os.path.join(models_dir, "imitation.pt"), captain, dims, heads, history)
print("saved", path)
if storage:
    storage.push(run_dir, None, status={"imitation_accuracy": dict(zip(names, history["test_accuracy"][-1]))})

## How good is the clone?

Before any RL: the cloned captains against each stage's opponent, next to random play. The stage notebooks then improve on this with reinforcement learning (and the commander).

In [ ]:
env = NavalEnv(stages="default", n_envs=config['n_envs'], unity_log_dir=os.path.join(run_dir, "unity_logs"))
model = MAPPO(env, **mappo_settings)
model.init_from_imitation(path)
rows = {}
for stage in range(6):
    clone, rand = evaluate(model, env, n_battles=24, stage=stage), evaluate(None, env, n_battles=24, stage=stage)
    rows[env.stages[stage]["name"]] = {"clone: pass rate": clone["pass_rate"], "clone: win rate": clone["win_rate"],
                                       "random: pass rate": rand["pass_rate"]}
env.close()
pd.DataFrame(rows).T

Next: [Stage 0 - Gunnery](Stage0-Gunnery.ipynb).